# Portugal – Presentation Visualisations (`04_visualizations_ppt.ipynb`)

Generates all figures for the group assignment presentation.

**Requires (from 02_modell.ipynb):**
- `network_unlimited.nc` — optimised network, no CO₂ limit
- `network_netzero.nc` — optimised network, net-zero CO₂
- `portugal_regions.gpkg` — region shapes

**Requires (from 03_sensitivity_analysis.ipynb):**
- `sensitivity_results.csv`

**Outputs of interest (assignment table):**
1. Total system cost split by technology
2. Capacity built per technology
3. Electricity mix (generation)
4. CO₂ shadow price
5. Price duration curves
6. Average electricity prices per region
7. Rate of curtailment
8. Storage filling levels throughout the year
9. Operation of the system (example week)
10. Sensitivity analysis summary

In [ ]:
import pypsa
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings("ignore")

plt.rcParams.update({"figure.dpi": 130, "font.size": 11})

COLORS = {
    "solar":                       "#f4a261",
    "onwind":                      "#457b9d",
    "offwind-float":               "#1d3557",
    "hydro":                       "#52b788",
    "CCGT":                        "#8d99ae",
    "coal":                        "#6d4c41",
    "biomass":                     "#a8dadc",
    "waste CHP":                   "#e9c46a",
    "battery storage":             "#2a9d8f",
    "hydrogen storage underground": "#e76f51",
    "AC":                          "#6c757d",
}
print("Setup complete.")

## Load networks

In [ ]:
n_free = pypsa.Network("network_unlimited.nc")
n_nz   = pypsa.Network("network_netzero.nc")
regions = gpd.read_file("portugal_regions.gpkg")

print(f"Unlimited network: {len(n_free.generators)} generators, {len(n_free.snapshots)} snapshots")
print(f"Net-zero network:  {len(n_nz.generators)} generators, {len(n_nz.snapshots)} snapshots")

---
## Figure 1 · Total System Cost split by technology

Shows how the total annualised system cost breaks down between conventional plants, renewables, storage, and transmission.

In [ ]:
def system_cost_by_carrier(net):
    """Return annualised capital + marginal cost per carrier [bn EUR/a]."""
    rows = []
    # Generators
    for name, g in net.generators.iterrows():
        dispatch = net.generators_t.p[name].multiply(net.snapshot_weightings.generators).sum()
        cap_cost = g.capital_cost * g.p_nom_opt
        mc = g.marginal_cost * dispatch
        rows.append({"carrier": g.carrier, "capital": cap_cost, "marginal": mc})
    # Storage
    for name, s in net.storage_units.iterrows():
        dispatch = net.storage_units_t.p[name].clip(lower=0).multiply(net.snapshot_weightings.generators).sum()
        rows.append({"carrier": s.carrier, "capital": s.capital_cost * s.p_nom_opt, "marginal": 0})
    # Links (transmission)
    for name, lk in net.links.iterrows():
        rows.append({"carrier": "AC", "capital": lk.capital_cost * lk.p_nom_opt, "marginal": 0})
    df = pd.DataFrame(rows).groupby("carrier")[["capital", "marginal"]].sum() / 1e9
    return df[df.sum(axis=1) > 0.001]

cost_free = system_cost_by_carrier(n_free)
cost_nz   = system_cost_by_carrier(n_nz)

all_carriers = sorted(set(cost_free.index) | set(cost_nz.index))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, cost_df, title in [
    (axes[0], cost_free, "No CO\u2082 limit"),
    (axes[1], cost_nz,   "Net-zero CO\u2082"),
]:
    cost_df = cost_df.reindex(all_carriers, fill_value=0)
    bottom_cap = np.zeros(len(all_carriers))
    bottom_mc  = np.zeros(len(all_carriers))
    for i, carrier in enumerate(all_carriers):
        c = COLORS.get(carrier, "#adb5bd")
        ax.bar(i, cost_df.at[carrier, "capital"], color=c, label=carrier if ax == axes[0] else "_")
        ax.bar(i, cost_df.at[carrier, "marginal"], bottom=cost_df.at[carrier, "capital"],
               color=c, alpha=0.4, hatch="//")
    ax.set_xticks(range(len(all_carriers)))
    ax.set_xticklabels(all_carriers, rotation=40, ha="right", fontsize=9)
    ax.set_ylabel("bn EUR/year")
    ax.set_title(f"System Cost – {title}", fontweight="bold")
    ax.grid(True, axis="y", alpha=0.3)
    total = cost_df.sum().sum()
    ax.set_title(f"System Cost – {title}\n(total: {total:.2f} bn EUR/a)", fontweight="bold")

axes[0].legend(bbox_to_anchor=(1.0, 1), loc="upper left", fontsize=8,
               title="Solid=capital, hatch=marginal")
plt.tight_layout()
plt.savefig("fig1_system_cost.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Figure 2 · Installed Capacity per technology

Comparison of optimally built capacity across both scenarios.

In [ ]:
def capacity_by_carrier(net):
    gen = net.generators.groupby("carrier")["p_nom_opt"].sum() / 1e3  # GW
    stor = net.storage_units.groupby("carrier")["p_nom_opt"].sum() / 1e3
    tx = pd.Series({"AC (transmission)": net.links["p_nom_opt"].sum() / 1e3})
    return pd.concat([gen, stor, tx])

cap_free = capacity_by_carrier(n_free)
cap_nz   = capacity_by_carrier(n_nz)
cap = pd.concat({"No CO\u2082 limit": cap_free, "Net-zero": cap_nz}, axis=1).fillna(0)
cap = cap[cap.sum(axis=1) > 0.01].sort_values("Net-zero", ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(cap))
w = 0.35
ax.barh(x - w/2, cap["No CO\u2082 limit"], w, label="No CO\u2082 limit",
        color=[COLORS.get(c.replace(" (transmission)", "AC"), "#adb5bd") for c in cap.index], alpha=0.7)
ax.barh(x + w/2, cap["Net-zero"], w, label="Net-zero",
        color=[COLORS.get(c.replace(" (transmission)", "AC"), "#adb5bd") for c in cap.index])
ax.set_yticks(x)
ax.set_yticklabels(cap.index)
ax.set_xlabel("Installed capacity [GW]")
ax.set_title("Optimal Installed Capacity by Technology", fontsize=13, fontweight="bold")
ax.legend()
ax.grid(True, axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig("fig2_capacity.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Figure 3 · Electricity Generation Mix

Annual electricity generation by technology [TWh], both scenarios.

In [ ]:
def generation_mix(net):
    gen = (net.generators_t.p
               .multiply(net.snapshot_weightings.generators, axis=0)
               .groupby(net.generators.carrier, axis=1).sum()
               .sum() / 1e6)  # TWh
    stor_dispatch = (net.storage_units_t.p.clip(lower=0)
                        .multiply(net.snapshot_weightings.generators, axis=0)
                        .groupby(net.storage_units.carrier, axis=1).sum()
                        .sum() / 1e6)
    return pd.concat([gen, stor_dispatch])

mix_free = generation_mix(n_free)
mix_nz   = generation_mix(n_nz)
mix = pd.concat({"No CO\u2082 limit": mix_free, "Net-zero": mix_nz}, axis=1).fillna(0)
mix = mix[mix.sum(axis=1) > 0.01]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, col in [(axes[0], "No CO\u2082 limit"), (axes[1], "Net-zero")]:
    data = mix[col].sort_values(ascending=False)
    data = data[data > 0]
    colors = [COLORS.get(c, "#adb5bd") for c in data.index]
    wedges, texts, autotexts = ax.pie(
        data.values, labels=data.index, colors=colors,
        autopct=lambda p: f"{p:.1f}%" if p > 2 else "",
        startangle=90, pctdistance=0.8,
    )
    total = data.sum()
    ax.set_title(f"{col}\nTotal: {total:.1f} TWh/a", fontweight="bold")
plt.suptitle("Annual Electricity Generation Mix", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("fig3_generation_mix.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Figure 4 · CO₂ Shadow Price

The dual variable (Lagrange multiplier) of the CO₂ constraint gives the marginal cost of decarbonisation in EUR/tCO₂.

In [ ]:
if "CO2Limit" in n_nz.global_constraints.index:
    co2_price = -n_nz.global_constraints.at["CO2Limit", "mu"]  # EUR/t CO2
    print(f"CO2 shadow price (net-zero): {co2_price:.1f} EUR/tCO2")
else:
    co2_price = None
    print("CO2Limit constraint not found in network (check that n_nz was optimised with the constraint)")

# For context: compare to carbon prices in other countries / ETS
fig, ax = plt.subplots(figsize=(6, 4))
if co2_price is not None:
    ax.bar(["Portugal\nNet-zero model"], [co2_price], color="#e63946", width=0.4)
    ax.axhline(100, color="grey", linestyle="--", label="EU ETS ~100 EUR/t (2024)")
    ax.set_ylabel("EUR / tCO\u2082")
    ax.set_title("CO\u2082 Shadow Price\n(marginal cost of net-zero)", fontweight="bold")
    ax.legend()
    ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("fig4_co2_shadow_price.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Figure 5 · Price Duration Curves

Marginal electricity price at each bus, sorted descending. Shows how often prices are high (scarcity) vs. low (renewable surplus).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, net, title in [
    (axes[0], n_free, "No CO\u2082 limit"),
    (axes[1], n_nz,   "Net-zero CO\u2082"),
]:
    prices = net.buses_t.marginal_price
    for bus in prices.columns:
        sorted_p = prices[bus].sort_values(ascending=False).values
        hours = np.arange(len(sorted_p)) * 3  # 3h resolution -> hours
        ax.plot(hours, sorted_p, label=bus, linewidth=1.5, alpha=0.85)
    ax.set_xlabel("Hours per year")
    ax.set_ylabel("Marginal price [EUR/MWh]")
    ax.set_title(f"Price Duration Curve – {title}", fontweight="bold")
    ax.legend(fontsize=8)
    ax.set_xlim(0, 8760)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(bottom=0)
plt.tight_layout()
plt.savefig("fig5_price_duration.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Figure 6 · Average Electricity Prices per Region

Time-averaged marginal price at each bus in both scenarios.

In [ ]:
avg_price_free = n_free.buses_t.marginal_price.mean()
avg_price_nz   = n_nz.buses_t.marginal_price.mean()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, avg_price, net, title in [
    (axes[0], avg_price_free, n_free, "No CO\u2082 limit"),
    (axes[1], avg_price_nz,   n_nz,   "Net-zero CO\u2082"),
]:
    merged = regions.set_index("region").join(avg_price.rename("price"))
    merged.plot(column="price", ax=ax, cmap="YlOrRd", legend=True,
                legend_kwds={"label": "EUR/MWh", "shrink": 0.6})
    for _, row in regions.iterrows():
        bus_price = avg_price.get(row["region"], np.nan)
        ax.annotate(f"{row['region']}\n{bus_price:.1f}",
                    (row["x"], row["y"]), ha="center", fontsize=8, fontweight="bold")
    ax.set_title(f"Average Electricity Price\n{title}", fontweight="bold")
    ax.set_axis_off()
plt.suptitle("Average Electricity Price per Region [EUR/MWh]", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("fig6_avg_prices.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Figure 7 · Rate of Curtailment

Curtailment = available renewable generation that is not dispatched. Ratio = curtailed / available.

In [ ]:
def curtailment_by_tech(net):
    rows = []
    re_techs = ["solar", "onwind", "offwind-float"]
    for name, g in net.generators.iterrows():
        if g.carrier not in re_techs:
            continue
        p_max = net.generators_t.p_max_pu[name] * g.p_nom_opt
        p_act = net.generators_t.p[name]
        w = net.snapshot_weightings.generators
        avail = (p_max * w).sum()
        actual = (p_act * w).sum()
        curt = avail - actual
        rows.append({"tech": g.carrier, "available": avail, "curtailed": curt})
    df = pd.DataFrame(rows).groupby("tech")[["available", "curtailed"]].sum()
    df["curtailment_rate"] = df["curtailed"] / df["available"].replace(0, np.nan)
    return df

curt_free = curtailment_by_tech(n_free)
curt_nz   = curtailment_by_tech(n_nz)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, curt, title in [
    (axes[0], curt_free, "No CO\u2082 limit"),
    (axes[1], curt_nz,   "Net-zero CO\u2082"),
]:
    rates = curt["curtailment_rate"].dropna() * 100
    bars = ax.bar(rates.index,
                  rates.values,
                  color=[COLORS.get(c, "#adb5bd") for c in rates.index])
    ax.bar_label(bars, fmt="%.1f%%", padding=3)
    ax.set_ylabel("Curtailment rate [%]")
    ax.set_title(f"Renewable Curtailment – {title}", fontweight="bold")
    ax.set_ylim(0, max(rates.max() * 1.3, 5))
    ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("fig7_curtailment.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Figure 8 · Storage Filling Levels Throughout the Year

Monthly average state of charge for battery and hydrogen storage (net-zero scenario).

In [ ]:
soc = n_nz.storage_units_t.state_of_charge.copy()
soc.columns = pd.MultiIndex.from_arrays(
    [n_nz.storage_units.carrier, soc.columns], names=["carrier", "unit"]
)

battery_soc = soc["battery storage"].sum(axis=1)
hydrogen_soc = soc["hydrogen storage underground"].sum(axis=1)

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

battery_monthly = battery_soc.resample("ME").mean() / 1e3  # GWh
hydrogen_monthly = hydrogen_soc.resample("ME").mean() / 1e3

axes[0].fill_between(battery_monthly.index, battery_monthly.values, alpha=0.7,
                      color=COLORS["battery storage"], label="Battery")
axes[0].set_ylabel("State of Charge [GWh]")
axes[0].set_title("Battery Storage – Monthly Average SoC (Net-zero)", fontweight="bold")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].fill_between(hydrogen_monthly.index, hydrogen_monthly.values, alpha=0.7,
                      color=COLORS["hydrogen storage underground"], label="Hydrogen")
axes[1].set_ylabel("State of Charge [GWh]")
axes[1].set_title("Hydrogen Storage – Monthly Average SoC (Net-zero)", fontweight="bold")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("fig8_storage_levels.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Figure 9 · System Operation – Example Week

A summer week (high solar) and a winter week (low solar, potential scarcity) in the net-zero scenario. Shows the interplay between supply, demand, and storage.

In [ ]:
def plot_dispatch_week(net, start, title, ax_list):
    """Plot dispatch + load + storage for one week on two axes."""
    snap = net.snapshots
    week = snap[(snap >= start) & (snap < pd.Timestamp(start) + pd.Timedelta("7D"))]
    
    # Generation by carrier
    gen = net.generators_t.p.loc[week]
    gen_by_carrier = gen.groupby(net.generators.carrier, axis=1).sum() / 1e3  # GW
    
    # Storage dispatch (positive = discharging)
    stor = net.storage_units_t.p.loc[week]
    stor_by_carrier = stor.groupby(net.storage_units.carrier, axis=1).sum() / 1e3
    
    # Total load
    load = net.loads_t.p_set.loc[week].sum(axis=1) / 1e3
    
    ax = ax_list[0]
    bottom = pd.Series(0.0, index=week)
    for carrier in gen_by_carrier.columns:
        if gen_by_carrier[carrier].abs().sum() < 0.01:
            continue
        ax.fill_between(week, bottom, bottom + gen_by_carrier[carrier],
                        label=carrier, color=COLORS.get(carrier, "#adb5bd"), alpha=0.9, step="post")
        bottom = bottom + gen_by_carrier[carrier]
    ax.plot(week, load, color="black", linewidth=2, label="Load", zorder=10)
    ax.set_ylabel("Power [GW]")
    ax.set_title(title, fontweight="bold")
    ax.legend(loc="upper right", fontsize=7, ncol=2)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(week[0], week[-1])
    
    ax2 = ax_list[1]
    for carrier in stor_by_carrier.columns:
        ax2.plot(week, stor_by_carrier[carrier], label=carrier,
                 color=COLORS.get(carrier, "#adb5bd"), linewidth=1.5)
    ax2.axhline(0, color="black", linewidth=0.8, linestyle="--")
    ax2.set_ylabel("Storage power [GW]\n(+discharge / -charge)")
    ax2.legend(loc="upper right", fontsize=7)
    ax2.grid(True, alpha=0.3)
    ax2.set_xlim(week[0], week[-1])

fig, axes = plt.subplots(4, 1, figsize=(14, 14))

plot_dispatch_week(n_nz, "2013-07-01", "Summer week (July 2013) – Net-zero", axes[0:2])
plot_dispatch_week(n_nz, "2013-01-07", "Winter week (January 2013) – Net-zero", axes[2:4])

plt.tight_layout()
plt.savefig("fig9_dispatch_weeks.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Figure 10 · Sensitivity Analysis Summary

Loads results from `03_sensitivity_analysis.ipynb`.

In [ ]:
df_sens = pd.read_csv("sensitivity_results.csv", index_col=0)
print(df_sens.round(2))

In [ ]:
x = df_sens.index.tolist()
tech_colors = {
    "Solar [GW]": COLORS["solar"],
    "Onshore wind [GW]": COLORS["onwind"],
    "Offshore wind [GW]": COLORS["offwind-float"],
    "CCGT [GW]": COLORS["CCGT"],
}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Cost
axes[0].plot(x, df_sens["System cost [bn EUR/a]"], marker="o", linewidth=2.5, color="steelblue")
for xi, yi in zip(x, df_sens["System cost [bn EUR/a]"]):
    axes[0].annotate(f"{yi:.2f}", (xi, yi), xytext=(0, 8), textcoords="offset points",
                     ha="center", fontsize=9)
axes[0].set_title("Total System Cost", fontsize=12, fontweight="bold")
axes[0].set_ylabel("bn EUR/year")
axes[0].set_xlabel("Solar PV capital cost")
axes[0].grid(True, alpha=0.3)

# Capacity stacked bar
bottom = np.zeros(len(x))
for col, color in tech_colors.items():
    vals = df_sens[col].values
    axes[1].bar(x, vals, bottom=bottom, label=col.replace(" [GW]", ""), color=color, alpha=0.9)
    bottom += vals
axes[1].set_title("Installed Capacity", fontsize=12, fontweight="bold")
axes[1].set_ylabel("GW")
axes[1].set_xlabel("Solar PV capital cost")
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3, axis="y")

# Storage & transmission
axes[2].plot(x, df_sens["Battery [GW]"], marker="s", label="Battery",
             color=COLORS["battery storage"], linewidth=2)
axes[2].plot(x, df_sens["Hydrogen [GW]"], marker="^", label="Hydrogen",
             color=COLORS["hydrogen storage underground"], linewidth=2)
axes[2].plot(x, df_sens["Transmission [GW]"], marker="D", label="Transmission",
             color=COLORS["AC"], linewidth=2)
axes[2].set_title("Storage & Transmission", fontsize=12, fontweight="bold")
axes[2].set_ylabel("GW")
axes[2].set_xlabel("Solar PV capital cost")
axes[2].legend(fontsize=8)
axes[2].grid(True, alpha=0.3)

plt.suptitle("Sensitivity Analysis: Solar PV Capital Cost (Net-Zero CO\u2082)",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("fig10_sensitivity.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Summary: Saved figures

| File | Content |
|------|---------|
| `fig1_system_cost.png` | Total system cost split by technology |
| `fig2_capacity.png` | Installed capacity per technology |
| `fig3_generation_mix.png` | Annual electricity generation mix |
| `fig4_co2_shadow_price.png` | CO₂ shadow price |
| `fig5_price_duration.png` | Price duration curves per region |
| `fig6_avg_prices.png` | Average electricity price per region (map) |
| `fig7_curtailment.png` | Rate of curtailment per renewable technology |
| `fig8_storage_levels.png` | Storage filling levels throughout the year |
| `fig9_dispatch_weeks.png` | System operation — summer & winter example weeks |
| `fig10_sensitivity.png` | Sensitivity analysis: solar cost variations |